In [43]:
import numpy as np
from scipy.signal import cheby2, filtfilt
from scipy.io import loadmat

In [44]:
def data_transformation(X, y, sfreq, tmax, tmin, tmin_original):
    print('Applying the desired time window.')
    beginning = int((tmin - tmin_original) * sfreq + 1)
    e = int((tmax - tmin_original) * sfreq)
    XX = X[:, :, beginning-1:e] # Python indexing starts at 0

    print('2D Reshaping: concatenating all 306 timeseries.')
    features = np.zeros((XX.shape[0], XX.shape[1] * XX.shape[2]), dtype=np.float32)
    for i in range(XX.shape[0]):
        temp = XX[i, :, :]
        features[i, :] = temp.flatten()

    print('Features Normalization.')
    for i in range(features.shape[1]):
        features[:, i] = features[:, i] - np.mean(features[:, i])
        features[:, i] = features[:, i] / np.std(features[:, i])

    print('Converting back to 3D matrix')
    features = np.reshape(features, (X.shape[0], X.shape[1], XX.shape[2]))
    features = np.transpose(features, (0, 1, 2))

    mTypeOne = []
    mTypeTwo = []
    f1 = np.where(y == 1)[0]
    f0 = np.where(y == 0)[0]

    for i in f1:
        mTypeOne.append(features[i, :, :])

    for i in f0:
        mTypeTwo.append(features[i, :, :])

    return mTypeOne, mTypeTwo


In [45]:

def nf_band_pass_filter(type_one, type_two, fs):
    # Define the frequency bands
    frequency = np.array([[4, 8], [8, 12], [12, 16], [16, 20]])
    print(frequency.shape)
    print(len(frequency),"len(frequency)")
    # Sampling frequency
    fs = fs
    filt_n = 2
    wn = np.array([x/(fs/2) for x in frequency])
    # wn = frequency[f] / (fs / 2)
    print(wn)

    # Preallocate filtered data arrays
    print(type_one.shape,"type_one shape")
    f_type_one = np.zeros_like(type_one)
    f_type_two = np.zeros_like(type_two)
    data_filter = np.zeros((len(frequency), type_one.shape[0], type_one.shape[1]))

    print(f_type_two.shape,"f_type_two")

    # Step 1: Applying Causal Chebyshev Type II Filter to each EEG Epoch
    for f in range(len(frequency)):
        print(wn[f])
        filter_b, filter_a = cheby2(filt_n * 2, 60, wn[f], 'bandpass')
        # Apply the filter to the data - using filtfilt for zero-phase filtering
        data_filter[f, :, :] = filtfilt(filter_b, filter_a, type_one, axis=1)
        f_type_one = data_filter[:,:,:]
        data_filter[f, :, :] = filtfilt(filter_b, filter_a, type_two, axis=1)
        f_type_two = data_filter[:,:,:]

    return f_type_one, f_type_two

In [46]:
def covariance(f_type_one, f_type_two):
    # Transposing 3D matrix
    type_one_t = np.transpose(f_type_one, (1, 0, 2))
    type_two_t = np.transpose(f_type_two, (1, 0, 2))

    # Dimensions
    Nch = f_type_one.shape[1]  # Number of channels
    Nf = f_type_one.shape[0]  # Number of frequency bands
    Nt = f_type_one.shape[2]  # Number of time samples

    # Initialize covariance matrices
    spe = 0
    spa = 0

    # Calculate the spatial & spectral covariance for fTypeOne
    for j in range(f_type_one.shape[2]):
        # Spectral covariance
        spe1 = np.dot(f_type_one[:, :, j], type_one_t[:, :, j])
        spe += spe1

        # Spatial covariance
        spa1 = np.dot(type_one_t[:, :, j], f_type_one[:, :, j])
        spa += spa1

    # Spectral covariance
    c_type_one_spectral = (1 / (Nf * Nt)) * spe
    # Spatial covariance
    c_type_one_spatial = (1 / (Nch * Nt)) * spa

    # Reset for fTypeTwo
    spe = 0
    spa = 0

    # Calculate the spatial & spectral covariance for fTypeTwo
    for j in range(f_type_two.shape[2]):
        # Spectral covariance
        spe1 = np.dot(f_type_two[:, :, j], type_two_t[:, :, j])
        spe += spe1

        # Spatial covariance
        spa1 = np.dot(type_two_t[:, :, j], f_type_two[:, :, j])
        spa += spa1

    # Spectral covariance
    c_type_two_spectral = (1 / (Nf * Nt)) * spe
    # Spatial covariance
    c_type_two_spatial = (1 / (Nch * Nt)) * spa

    return c_type_one_spectral, c_type_one_spatial, c_type_two_spectral, c_type_two_spatial

In [59]:
def eigenvalues_problem(c_type_one_spectral, c_type_one_spatial, c_type_two_spectral, c_type_two_spatial, f_type_one, f_type_two, d):
    Nch = c_type_one_spatial.shape[0]
    Nf = c_type_one_spectral.shape[0]
    print(Nch,"Nch")
    print(Nf,"Nf")
    lambda_m = np.zeros((Nf, Nch))
    lambda_n = np.zeros((Nf, Nch))

    # Spatial Eigenvalues
    wr1, vr1 = np.linalg.eig(np.linalg.solve(c_type_one_spatial, c_type_one_spatial + c_type_two_spatial))
    wr2, vr2 = np.linalg.eig(np.linalg.solve(c_type_two_spatial, c_type_two_spatial + c_type_one_spatial))
    vr1 = np.diag(vr1)
    vr2 = np.diag(vr2)

    # Spectral Eigenvalues
    wl1, vl1 = np.linalg.eig(np.linalg.solve(c_type_one_spectral, c_type_one_spectral + c_type_two_spectral))
    wl2, vl2 = np.linalg.eig(np.linalg.solve(c_type_two_spectral, c_type_two_spectral + c_type_one_spectral))
    vl1 = np.diag(vl1)
    vl2 = np.diag(vl2)



    for p in range(Nf):
        for q in range(Nch):
            lambda_m[p, q] = (vl1[p] * vr1[q]) / (vl1[p] * vr1[q] + (1 - vl1[p]) * (1 - vr1[q]))
            lambda_n[p, q] = (vl2[p] * vr2[q]) / (vl2[p] * vr2[q] + (1 - vl2[p]) * (1 - vr2[q]))
            # lambda_m[p,q] = (vl1[p,1]*vr1[q,1])/(vl1[p,1]*vr1[q,1] + (1-vl1[p,1])*(1-vr1[q,1]))
            # lambda_n[p,q] = (vl2[p,1]*vr2[q,1])/(vl2[p,1]*vr2[q,1] + (1-vl2[p,1])*(1-vr2[q,1]))

    lambda_k1 = np.sort(lambda_m, axis=None)[::-1]
    lambda_k2 = np.sort(lambda_n, axis=None)[::-1]

    w1 = np.kron(wr1, wl1)  # Kronecker Product
    w2 = np.kron(wr2, wl2)

    # y1 = np.dot(w1.T, f_type_one.reshape(-1, f_type_one.shape[2]))
    # y2 = np.dot(w2.T, f_type_two.reshape(-1, f_type_two.shape[2]))

    print(w1.shape)

    y1 = np.zeros((w1.shape[0], 1, f_type_one.shape[2]))
    y2 = np.zeros((w2.shape[0], 1, f_type_two.shape[2]))

    for i in range(f_type_one.shape[2]):
        y1[:, :, i] = np.dot(w1.T, f_type_one[:, :, i].reshape(-1, 1))
        y2[:, :, i] = np.dot(w2.T, f_type_two[:, :, i].reshape(-1, 1))
    print(y1.shape,y2.shape,"y1 y2 shape")

    # Removing Singleton Dimension
    y1 = np.squeeze(y1[:d, :])
    y2 = np.squeeze(y2[:d, :])
    print(y1.shape,y2.shape,"y1 y2 shape squeeze")

    var_y1 = np.var(y1, axis=0)
    var_y2 = np.var(y2, axis=0)

    # Sum of variances
    sum_var_y1 = np.sum(var_y1)
    sum_var_y2 = np.sum(var_y2)

    # Calculate Z1 and Z2
    z1 = np.log(var_y1 / sum_var_y1).T
    z2 = np.log(var_y2 / sum_var_y2).T

    z1 = z1.reshape(-1, 1)
    z2 = z2.reshape(-1, 1)

    print(z1.shape,z2.shape,"z1 z2 shape size")

    return z1, z2

In [60]:
X = np.random.rand(100, 64, 250)
Y = np.hstack((np.ones(50), np.zeros(50)))
np.random.shuffle(Y)
print(Y)

# tmin = 0
# tmax = 0.5
fs = 250
d = 5 # Number of discriminant features
window = 4
number_of_samples = fs*window

# m_type_one, m_type_two = data_transformation(X, Y, fs, tmax, tmin, -0.5)


m_type_one = np.random.rand(50, 64, number_of_samples)
m_type_two = np.random.rand(50, 64, number_of_samples)



# Lists to store features from each epoch
type_one_z = []
type_two_z = []
print(len(m_type_one),"len(m_type_one)")


# Processing each epoch
for j in range(len(m_type_one)):
    type_one = m_type_one[j]
    type_two = m_type_two[j]

    print('Step 1: Applying Causal Chebyshev Type II Filter to each EEG Epoch')
    f_type_one, f_type_two = nf_band_pass_filter(type_one, type_two, fs)
    print(f_type_one.shape,"ftype shape")
    print('Step 2: Calculate the spectral and spatial covariance')
    c_type_one_spectral, c_type_one_spatial, c_type_two_spectral, c_type_two_spatial = covariance(f_type_one, f_type_two)
    print(c_type_two_spectral.shape,c_type_two_spatial.shape,"c_type_one_spectral","c_type_one_spatial")
    print('Step 3: Solve generalized eigenvalue problem and extract the feature vector')
    z1, z2 = eigenvalues_problem(c_type_one_spectral, c_type_one_spatial, c_type_two_spectral, c_type_two_spatial, f_type_one, f_type_two, d)
    print(z1.shape,z2.shape,"z1,z1")
    type_one_z.append(z1)
    type_two_z.append(z2)


type_one_z = np.array(type_one_z)
type_two_z = np.array(type_two_z)

print(type_one_z.shape,type_two_z.shape,"type z")

# Preparing the dataset
cov_one = np.stack(type_one_z, axis=-1)
cov_two = np.stack(type_two_z, axis=-1)

# Formatting Dataset
k = np.concatenate([cov_one, cov_two], axis=-1)  # Concatenating along the last axis
print(k.shape,"k shape")
x = k.reshape(k.shape[-1], -1)  # Reshaping to 2D array where each row is an epoch
print(x.shape)

# Labels for Class 1 = 0, Class 2 = 1
y = np.concatenate([np.ones(cov_one.shape[-1]), np.zeros(cov_two.shape[-1])])



[1. 0. 0. 1. 1. 1. 0. 0. 0. 1. 1. 1. 0. 1. 0. 1. 0. 0. 1. 0. 1. 1. 0. 0.
 0. 0. 1. 0. 1. 1. 0. 0. 1. 0. 1. 1. 1. 1. 0. 0. 1. 1. 0. 0. 0. 1. 0. 1.
 1. 0. 1. 0. 1. 1. 1. 0. 1. 1. 0. 0. 1. 0. 0. 1. 1. 0. 1. 0. 0. 0. 1. 1.
 0. 0. 0. 0. 1. 0. 0. 1. 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 1. 0. 1. 0. 1.
 0. 1. 1. 1.]
50 len(m_type_one)
Step 1: Applying Causal Chebyshev Type II Filter to each EEG Epoch
(4, 2)
4 len(frequency)
[[0.032 0.064]
 [0.064 0.096]
 [0.096 0.128]
 [0.128 0.16 ]]
(64, 1000) type_one shape
(64, 1000) f_type_two
[0.032 0.064]
[0.064 0.096]
[0.096 0.128]
[0.128 0.16 ]
(4, 64, 1000) ftype shape
Step 2: Calculate the spectral and spatial covariance
(4, 4) (64, 64) c_type_one_spectral c_type_one_spatial
Step 3: Solve generalized eigenvalue problem and extract the feature vector
64 Nch
4 Nf
(256,)


/tmp/ipykernel_21092/4246174118.py:25: ComplexWarning: Casting complex values to real discards the imaginary part
  lambda_m[p, q] = (vl1[p] * vr1[q]) / (vl1[p] * vr1[q] + (1 - vl1[p]) * (1 - vr1[q]))
/tmp/ipykernel_21092/4246174118.py:26: ComplexWarning: Casting complex values to real discards the imaginary part
  lambda_n[p, q] = (vl2[p] * vr2[q]) / (vl2[p] * vr2[q] + (1 - vl2[p]) * (1 - vr2[q]))
/tmp/ipykernel_21092/4246174118.py:45: ComplexWarning: Casting complex values to real discards the imaginary part
  y1[:, :, i] = np.dot(w1.T, f_type_one[:, :, i].reshape(-1, 1))
/tmp/ipykernel_21092/4246174118.py:46: ComplexWarning: Casting complex values to real discards the imaginary part
  y2[:, :, i] = np.dot(w2.T, f_type_two[:, :, i].reshape(-1, 1))
/tmp/ipykernel_21092/4246174118.py:62: RuntimeWarning: divide by zero encountered in log
  z1 = np.log(var_y1 / sum_var_y1).T
/tmp/ipykernel_21092/4246174118.py:63: RuntimeWarning: divide by zero encountered in log
  z2 = np.log(var_y2 / su

(256, 1, 1000) (256, 1, 1000) y1 y2 shape
(5, 1000) (5, 1000) y1 y2 shape squeeze
(1000, 1) (1000, 1) z1 z2 shape size
(1000, 1) (1000, 1) z1,z1
Step 1: Applying Causal Chebyshev Type II Filter to each EEG Epoch
(4, 2)
4 len(frequency)
[[0.032 0.064]
 [0.064 0.096]
 [0.096 0.128]
 [0.128 0.16 ]]
(64, 1000) type_one shape
(64, 1000) f_type_two
[0.032 0.064]
[0.064 0.096]
[0.096 0.128]
[0.128 0.16 ]
(4, 64, 1000) ftype shape
Step 2: Calculate the spectral and spatial covariance
(4, 4) (64, 64) c_type_one_spectral c_type_one_spatial
Step 3: Solve generalized eigenvalue problem and extract the feature vector
64 Nch
4 Nf
(256,)
(256, 1, 1000) (256, 1, 1000) y1 y2 shape
(5, 1000) (5, 1000) y1 y2 shape squeeze
(1000, 1) (1000, 1) z1 z2 shape size
(1000, 1) (1000, 1) z1,z1
Step 1: Applying Causal Chebyshev Type II Filter to each EEG Epoch
(4, 2)
4 len(frequency)
[[0.032 0.064]
 [0.064 0.096]
 [0.096 0.128]
 [0.128 0.16 ]]
(64, 1000) type_one shape
(64, 1000) f_type_two
[0.032 0.064]
[0.064 0.0

In [ ]:
def fit_scssp(n_components,fs,d,c1,c2):


    m_type_one = c1
    m_type_two = c2



    # Lists to store features from each epoch
    type_one_z = []
    type_two_z = []
    print(len(m_type_one),"len(m_type_one)")


    # Processing each epoch
    for j in range(len(m_type_one)):
        type_one = m_type_one[j]
        type_two = m_type_two[j]

        print('Step 1: Applying Causal Chebyshev Type II Filter to each EEG Epoch')
        f_type_one, f_type_two = nf_band_pass_filter(type_one, type_two, fs)
        print(f_type_one.shape,"ftype shape")
        print('Step 2: Calculate the spectral and spatial covariance')
        c_type_one_spectral, c_type_one_spatial, c_type_two_spectral, c_type_two_spatial = covariance(f_type_one, f_type_two)
        print(c_type_two_spectral.shape,c_type_two_spatial.shape,"c_type_one_spectral","c_type_one_spatial")
        print('Step 3: Solve generalized eigenvalue problem and extract the feature vector')
        z1, z2 = eigenvalues_problem(c_type_one_spectral, c_type_one_spatial, c_type_two_spectral, c_type_two_spatial, f_type_one, f_type_two, d)
        print(z1.shape,z2.shape,"z1,z1")
        type_one_z.append(z1)
        type_two_z.append(z2)


    type_one_z = np.array(type_one_z)
    type_two_z = np.array(type_two_z)

    print(type_one_z.shape,type_two_z.shape,"type z")

    # Preparing the dataset
    cov_one = np.stack(type_one_z, axis=-1)
    cov_two = np.stack(type_two_z, axis=-1)

    # Formatting Dataset
    k = np.concatenate([cov_one, cov_two], axis=-1)  # Concatenating along the last axis
    print(k.shape,"k shape")
    x = k.reshape(k.shape[-1], -1)  # Reshaping to 2D array where each row is an epoch
    print(x.shape)

    # Labels for Class 1 = 0, Class 2 = 1
    y = np.concatenate([np.ones(cov_one.shape[-1]), np.zeros(cov_two.shape[-1])])

